# 1. Setup Spark

In [7]:
!pip install pyspark

  Using cached pyspark-4.1.1.tar.gz (455.4 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008706 sha256=90617a336e66f438aa165faf45a4cacb9b00b74c3d2b676ff6efe7284f3288f9
  Stored in directory: /Users/enfants/Library/Caches/pip/wheels/f4/ca/ea/203f40b3e935bbf99bee851c2f4a87d22996ab8212d367ce58
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


In [23]:
import pyspark
import pyspark.sql
import panda as pd

ModuleNotFoundError: No module named 'panda'

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Predicting U.S. Flight Delay") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/06 18:50:33 WARN Utils: Your hostname, MacBook-Air-de-Nicolas-3.local, resolves to a loopback address: 127.0.0.1; using 192.0.0.2 instead (on interface en8)
26/03/06 18:50:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 18:50:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# 2. Data Loading

In [10]:
flights = spark.read.csv(
    "Data/Flights/*.csv",
    header=True,
    inferSchema=True
)

flights.head()

26/03/06 18:51:53 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: Data/Flights/*.csv.
java.io.FileNotFoundException: File Data/Flights/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

Row(FL_DATE=datetime.date(2013, 7, 1), OP_CARRIER_AIRLINE_ID=20363, OP_CARRIER_FL_NUM=3407, ORIGIN_AIRPORT_ID=11433, DEST_AIRPORT_ID=13342, CRS_DEP_TIME=1040, ARR_DELAY_NEW=0.0, CANCELLED=0.0, DIVERTED=0.0, CRS_ELAPSED_TIME=79.0, WEATHER_DELAY=None, NAS_DELAY=None, _c12=None)

In [12]:
flights.printSchema()
flights.show(5)

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER_AIRLINE_ID: integer (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- ARR_DELAY_NEW: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- WEATHER_DELAY: double (nullable = true)
 |-- NAS_DELAY: double (nullable = true)
 |-- _c12: string (nullable = true)

+----------+---------------------+-----------------+-----------------+---------------+------------+-------------+---------+--------+----------------+-------------+---------+----+
|   FL_DATE|OP_CARRIER_AIRLINE_ID|OP_CARRIER_FL_NUM|ORIGIN_AIRPORT_ID|DEST_AIRPORT_ID|CRS_DEP_TIME|ARR_DELAY_NEW|CANCELLED|DIVERTED|CRS_ELAPSED_TIME|WEATHER_DELAY|NAS_DELAY|_c12|
+----------+---------------------+-----------------+----

26/03/06 18:52:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: FL_DATE, OP_CARRIER_AIRLINE_ID, OP_CARRIER_FL_NUM, ORIGIN_AIRPORT_ID, DEST_AIRPORT_ID, CRS_DEP_TIME, ARR_DELAY_NEW, CANCELLED, DIVERTED, CRS_ELAPSED_TIME, WEATHER_DELAY, NAS_DELAY, 
 Schema: FL_DATE, OP_CARRIER_AIRLINE_ID, OP_CARRIER_FL_NUM, ORIGIN_AIRPORT_ID, DEST_AIRPORT_ID, CRS_DEP_TIME, ARR_DELAY_NEW, CANCELLED, DIVERTED, CRS_ELAPSED_TIME, WEATHER_DELAY, NAS_DELAY, _c12
Expected: _c12 but found: 
CSV file: file:///Users/enfants/Code/Predicting%20U.S.%20Flight%20Delays/Data/Flights/201307.csv


In [13]:
weather = spark.read.csv(
    "Data/Weather/*.txt",
    header=True,
    inferSchema=True
)

26/03/06 18:53:06 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: Data/Weather/*.txt.
java.io.FileNotFoundException: File Data/Weather/*.txt does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

In [14]:
weather.printSchema()
weather.show(5)

root
 |-- WBAN: integer (nullable = true)
 |-- Date: integer (nullable = true)
 |-- Time: integer (nullable = true)
 |-- StationType: integer (nullable = true)
 |-- SkyCondition: string (nullable = true)
 |-- SkyConditionFlag: string (nullable = true)
 |-- Visibility: string (nullable = true)
 |-- VisibilityFlag: string (nullable = true)
 |-- WeatherType: string (nullable = true)
 |-- WeatherTypeFlag: string (nullable = true)
 |-- DryBulbFarenheit: string (nullable = true)
 |-- DryBulbFarenheitFlag: string (nullable = true)
 |-- DryBulbCelsius: string (nullable = true)
 |-- DryBulbCelsiusFlag: string (nullable = true)
 |-- WetBulbFarenheit: string (nullable = true)
 |-- WetBulbFarenheitFlag: string (nullable = true)
 |-- WetBulbCelsius: string (nullable = true)
 |-- WetBulbCelsiusFlag: string (nullable = true)
 |-- DewPointFarenheit: string (nullable = true)
 |-- DewPointFarenheitFlag: string (nullable = true)
 |-- DewPointCelsius: string (nullable = true)
 |-- DewPointCelsiusFlag: str

26/03/06 18:53:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [15]:
print("Flights:", flights.count())
print("Weather:", weather.count())

Flights: 18286055


Weather: 32631312


# 3. Filtering

In [17]:
FT = flights.filter((flights.CANCELLED == 0) & (flights.DIVERTED == 0))

In [18]:
print("FT:", FT.count())

FT: 17943069


In [19]:
wban_airport_timezone = spark.read.csv(
    "Data/wban_airport_timezone.csv",
    header=True,
    inferSchema=True
)

In [21]:
wban_airport_timezone.printSchema()
wban_airport_timezone.show(5)

root
 |-- AirportID: integer (nullable = true)
 |-- WBAN: integer (nullable = true)
 |-- TimeZone: integer (nullable = true)

+---------+-----+--------+
|AirportID| WBAN|TimeZone|
+---------+-----+--------+
|    10685|54831|      -6|
|    14871|24232|      -8|
|    10620|24033|      -7|
|    14747|24233|      -8|
|    11252|12834|      -5|
+---------+-----+--------+
only showing top 5 rows


In [26]:
wban = wban_airport_timezone.select("WBAN").distinct()
OT = weather.filter(weather.WBAN == wban)

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `_get_object_id` is not supported.

26/03/07 02:46:46 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 951459 ms exceeds timeout 120000 ms
26/03/07 02:46:46 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/07 03:02:19 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$